# CCM-v3 — corrected SmolLM2-360M layer 0: RoPE + grouped heads + pretrained initialized cellular consensus

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/research/competitive-cenn-feasibility/notebooks/CCM_v3_Smol360_RoPE_Parity_Chat_Colab.ipynb)

**Why v2 failed:** `CCM_v2` discarded pretrained Q/K/V weights, lost rotary positional encoding (RoPE), and compressed all 15 query / 5 KV heads into one content-addressed pool. Its real gold next-token metrics improved with training, but chat degenerated into repeated digits. 

**v3 fixes the interface and initialization first:** the native trained projections are **copied and frozen**; RoPE, grouped heads, causal masking, and the original output projection are retained; the readout uses only **pairwise local log-mass conserving cellular merges**, not the native attention module or SDPA. Trainable low-rank correction begins at zero.

**Required precondition:** two independent full-model checks **before training**: an FP32 algebraic fidelity control (temporary full-model FP32 conversion, then restoration to the normal GPU inference precision) and a reduced-precision drift control (mean absolute logits, relative MSE and argmax agreement). An isolated maximum logit over millions of values is **not** a sufficient failure criterion on FP16/BF16. Stop for genuine FP32 miswiring or excessive reduced-precision divergence. Diagnostics are saved even if the gate stops. The corrected model is **mathematically attention-equivalent on its bounded tape**, *not* a novel attention-free or faster architecture. Old CCM-v2 checkpoints are **not compatible** with this new structure. No KV cache: chat is greedy full-prefix recomputation.

In [ ]:
# @title 1. Install pinned environment and clone active research branch
%pip -q install "transformers==4.57.6" "datasets>=3,<5" "huggingface_hub>=0.36,<2" "accelerate>=1" "safetensors" "pytest>=8" "pandas" "matplotlib"
import sys,subprocess,json,hashlib,pathlib,torch,os
from pathlib import Path
REPO=Path('/content/TinyCeNN-LM')
BRANCH='research/competitive-cenn-feasibility'
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO));sys.path.insert(0,str(REPO/'src'))
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print('Pinned Git commit:',GIT_SHA,'torch:',torch.__version__)
if not torch.cuda.is_available():raise RuntimeError('Set Runtime > Change runtime type > NVIDIA GPU (T4 or better)')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# @title 2. CPU operator tests: exact causal SDPA equivalence + RoPE + grouped heads
subprocess.run([sys.executable,'-m','pytest','-q',
                str(REPO/'tests/test_ccm_v3_rotary_consensus.py'),
                str(REPO/'tests/test_ccm_v3_parity_preflight.py')],check=True)
print('NOTE: These only test the layer algebra. The next cell checks REAL full-model parity.')

## 3 · Model + training controls

Start with `smoke` and `STEPS=0` to establish strict FP32 algebraic parity and acceptable runtime precision drift. A prior T4 run had full-model max absolute logit difference 0.4834 but relative MSE 0.0003007 and perfect token agreement, so we now test the *cause* rather than rejecting one outlier. Then use `quick` (optional 64 steps, rank-16 output correction) or `research`. **The old v2 checkpoint cannot be resumed**; new v3 checkpoints from this notebook can be resumed only with an exact configuration/revision match. Training never optimizes teacher agreement.

In [ ]:
# @title 3. Research settings
PROFILE='smoke'  # @param ['smoke','quick','research']
SLOTS=256  # @param {type:'integer'}
QUERY_BLOCK=16 # @param {type:'integer'}
CORRECTION_RANK=16 # @param {type:'integer'}
STEPS=0  # @param {type:'integer'}
# -1 = profile default; 0 = parity-only without extra optimization
MAX_FP32_ABS_LOGITS=0.02 # @param {type:'number'}
MAX_FP32_REL_MSE=1e-6  # true FP32 algebraic parity
MAX_RUNTIME_REL_MSE=0.001 # @param {type:'number'}
MAX_RUNTIME_MEAN_ABS=0.08 # @param {type:'number'}
LEARNING_RATE=0.0002 # @param {type:'number'}
ARC_QUESTIONS=16 # @param {type:'integer'}
CHAT_NEW_TOKENS=48 # @param {type:'integer'}
SKIP_CHAT=False # @param {type:'boolean'}
SKIP_ARC=False # @param {type:'boolean'}
AUTO_DOWNLOAD=True # @param {type:'boolean'}
RESUME_V3_CHECKPOINT='' # @param {type:'string'}
# If resuming a v3 model, provide a local path to best_ccm_v3_layer0.pt, never a v2 checkpoint.
OUTPUT_DIR=Path('/content/CCM_v3_Smol360_RoPE_Parity_Results')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert SLOTS>=2 and SLOTS&(SLOTS-1)==0
print('Profile',PROFILE,'memory slots',SLOTS,'steps',STEPS,'output',OUTPUT_DIR)

In [ ]:
# @title 4. Native-initialized parity test → optional real-label training → sealed TEST + ARC-Easy + chat + GPU timing
# Reload source in a live Colab kernel after git fetch (avoid stale old parity gate).
import importlib
import scripts.benchmark_ccm_v3_smol360 as ccm_v3_benchmark
importlib.reload(ccm_v3_benchmark)
main=ccm_v3_benchmark.main
args=['--profile',PROFILE,'--slots',str(SLOTS),'--query-block',str(QUERY_BLOCK),
      '--correction-rank',str(CORRECTION_RANK),'--lr',str(LEARNING_RATE),
      '--arc-questions',str(ARC_QUESTIONS),'--chat-new-tokens',str(CHAT_NEW_TOKENS),
      '--output-dir',str(OUTPUT_DIR),
      '--max-abs-parity',str(MAX_FP32_ABS_LOGITS),
      '--max-fp32-rel-mse',str(MAX_FP32_REL_MSE),
      '--max-runtime-rel-mse',str(MAX_RUNTIME_REL_MSE),
      '--max-runtime-mean-abs',str(MAX_RUNTIME_MEAN_ABS)]
if STEPS>=0:args+=['--steps',str(STEPS)]
if RESUME_V3_CHECKPOINT:args+=['--resume-checkpoint',RESUME_V3_CHECKPOINT]
if SKIP_CHAT:args+=['--skip-chat']
if SKIP_ARC:args+=['--skip-arc']
try:
    report,archive,session=main(args,return_runtime=True)
except Exception as error:
    # Always leave a portable diagnostic ZIP even if the strict FP32 check fails.
    import zipfile,traceback
    (OUTPUT_DIR/'colab_error.txt').write_text(traceback.format_exc())
    initial=OUTPUT_DIR/'initial_parity.json'
    if initial.exists():
        print('STRICT PARITY DIAGNOSTIC:',initial.read_text())
    fail_zip=OUTPUT_DIR.parent/(OUTPUT_DIR.name+'_PARITY_FAILURE.zip')
    with zipfile.ZipFile(fail_zip,'w',zipfile.ZIP_DEFLATED) as z:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path.is_file():z.write(path,path.relative_to(OUTPUT_DIR))
    print('PARITY FAILURE ZIP:',fail_zip)
    raise
print('RUNTIME PARITY (native reduced precision):',json.dumps(report['parity'],indent=2))
print('FP32 ALGEBRAIC PARITY:',json.dumps(report['fp32_parity'],indent=2))
print('PARITY DECISION:',json.dumps(report['parity_decision'],indent=2))
print('REAL gold DEV ΔNLL:',report['dev']['delta_nll'])
print('REAL gold TEST ΔNLL:',report['test']['delta_nll'])
print('REAL gold TEST accuracy native/cellular:',report['test']['teacher_gold_accuracy'],
      report['test']['student_gold_accuracy'])
print('ARC labels:',report['arc_score'])
print('Chat rubric:',report['chat_score'])
print('GPU full-model no-cache speedup:',report['paired_latency'].get('speedup'))
print('Native pretrained projection parameters:',report['native_params'],
      'CCM-v3:',report['student_params'])
print('ATTENTION-EQUIVALENT research control, NOT novel attention-free or a verified speedup.')

In [ ]:
# @title 5. Plot gold DEV training curve and inspect results
import pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
history=pd.read_csv(OUTPUT_DIR/'dev_history.csv')
display(history)
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(history['step'],history['dev_gold_nll'],label='Cellular gold DEV NLL',marker='o')
ax.plot(history['step'],history['native_gold_nll'],label='Original native gold DEV NLL',ls='--')
ax.set(xlabel='Training step',ylabel='Gold NLL',title='CCM-v3 position-aware head-parity control')
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_DIR/'dev_gold_nll.png',dpi=140);plt.show()
if (OUTPUT_DIR/'chat_comparison.json').exists():
    obj=json.loads((OUTPUT_DIR/'chat_comparison.json').read_text())
    display(pd.DataFrame([{'test':a['id'],'prompt':a['prompt'],
       'native_correct':a['pass'],'ccm_v3_correct':b['pass'],
       'native_answer':a['response'],'ccm_v3_answer':b['response']}
       for a,b in zip(obj['native']['rows'],obj['ccm_v3']['rows'])]))

In [ ]:
# @title 6. Interactive custom chat; same prompt, native vs cellular (no cache)
CUSTOM_QUESTION='' # @param {type:'string'}
if CUSTOM_QUESTION.strip():
    from scripts.real_fast_eval import greedy_no_cache
    from scripts.benchmark_ccm_consensus_smol360 import attention_slot
    mdl,native,cell,tokenizer=(session[k] for k in ('model','native','student','tokenizer'))
    with attention_slot(mdl,native):answer_native=greedy_no_cache(mdl,tokenizer,CUSTOM_QUESTION,max_new_tokens=CHAT_NEW_TOKENS)
    with attention_slot(mdl,cell):answer_v3=greedy_no_cache(mdl,tokenizer,CUSTOM_QUESTION,max_new_tokens=CHAT_NEW_TOKENS)
    print('NATIVE:',answer_native['response'])
    print('CELLULAR:',answer_v3['response'])
else:
    print('Enter your own question, then rerun this cell.')

In [ ]:
# @title 7. Export training checkpoint, parity, chat, ARC and speed statistics as ZIP
import zipfile
bundle=OUTPUT_DIR.parent/(OUTPUT_DIR.name+'_full.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():z.write(path,path.relative_to(OUTPUT_DIR))
print('Results archive:',bundle,'bytes',bundle.stat().st_size)
if AUTO_DOWNLOAD:
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as ex:
        print('Download manually from Files:',bundle,'error:',str(ex))

## Interpretation

**A successful native-parity control is not proof of a new attention alternative.** The conservation rule is mathematically softmax attention over the bounded cell window. The full 360M model also retains the other 31 native Transformer attention blocks. CCM-v3 retains full QKV/O projection parameters and can be slower than native SDPA. Its chief purpose is to correct the scientifically broken layer interface and establish a stable, controlled reference before testing compression, non-attention cellular dynamics, or hardware implementations.

The earlier v2 step-80 checkpoint cannot be loaded into v3. The preflight checks strict FP32 algebraic accuracy and realistic low-precision drift *separately*. If either fails, it saves `initial_parity.json` plus a downloadable failure ZIP and **stops rather than training a miswired module**. A tolerance passing the T4 reported FP16 measurement does not automatically pass the strict FP32 control.